# Test: Online CP + Normalized NCM

This notebook tests whether combining online calibration (Experiment 2) with normalized NCM (Experiment 3) improves coverage beyond what either achieves alone.

**Approach**: Use the existing `get_normalized_prediction_intervals()` function inside a daily online loop — no new functions needed.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import xgboost as xgb
from utils.conformal import get_normalized_prediction_intervals, get_fast_prediction_intervals
from utils.evaluation import compute_picp, compute_mpiw, compute_calibration_error, compute_winkler_score
from utils.temporal_splits import get_temporal_split_by_period
from utils.data_loading import load_cached_dataframe

TARGET_COVERAGE = 0.90
ALPHA = 0.10

# Load route-level features
df = load_cached_dataframe('../outputs/processed_data/route_features.parquet')

# Load model
model = xgb.XGBRegressor()
model.load_model('../outputs/models/route_xgboost_model.json')

# Feature columns
feature_cols = ['hour_of_day', 'hour_sin', 'hour_cos', 'day_of_week', 'dow_sin', 'dow_cos',
                'is_weekend', 'minute_of_day', 'route_id_encoded', 'direction_encoded',
                'hist_route_mean', 'hist_route_std', 'hist_route_median',
                'hist_route_q25', 'hist_route_q75', 'hist_route_count']
TARGET_COL = 'total_travel_time_seconds'

# Temporal split
splits = get_temporal_split_by_period(df, exclude_anomalous=True)
cal_df = splits['calibration']

# Combine test periods
test_dfs = []
for period in ['test_near', 'test_mid', 'test_far']:
    tmp = splits[period].copy()
    tmp['period'] = period
    test_dfs.append(tmp)
test_df = pd.concat(test_dfs).sort_values('date')

X_cal = cal_df[feature_cols].values
y_cal = cal_df[TARGET_COL].values
X_test = test_df[feature_cols].values
y_test = test_df[TARGET_COL].values
dates_test = test_df['date'].values

print(f'Calibration: {len(y_cal)} samples')
print(f'Test: {len(y_test)} samples')
print(f'Test dates: {len(np.unique(dates_test))} unique days')

  train: 7,598 records (21 days)
  calibration: 2,740 records (7 days)
  test_near: 2,707 records (7 days)
  test_mid: 1,833 records (5 days)
  test_far: 4,736 records (13 days)
Calibration: 2740 samples
Test: 9276 samples
Test dates: 25 unique days


## Method: Online Loop with Normalized NCM

For each day:
1. Call `get_normalized_prediction_intervals()` with current calibration set (no segment_ids → uses prediction-magnitude quintiles for sigma)
2. Record predictions and intervals
3. Append revealed true values to calibration set
4. Apply window if specified

This is identical to Online CP except `get_normalized_prediction_intervals()` replaces `get_fast_prediction_intervals()`.

In [2]:
from tqdm import tqdm

def online_cp_with_normalized_ncm(model, X_stream, y_stream, X_cal_init, y_cal_init,
                                   dates_stream, confidence=0.90, window_size=None):
    """Online CP using normalized NCM (adaptive widths) instead of absolute NCM."""
    X_cal = np.copy(X_cal_init)
    y_cal = np.copy(y_cal_init)
    
    y_pred_all = []
    lower_all = []
    upper_all = []
    
    unique_dates = sorted(set(dates_stream))
    
    for date in tqdm(unique_dates, desc='Online Normalized NCM'):
        # Get batch for this day
        mask = dates_stream == date
        X_batch = X_stream[mask]
        y_batch = y_stream[mask]
        
        # Use normalized NCM (no segment_ids → uses quintile-based sigma)
        y_pred, lower, upper = get_normalized_prediction_intervals(
            model=model,
            X_cal=X_cal,
            y_cal=y_cal,
            X_test=X_batch,
            segment_ids_cal=None,  # No segments → quintile-based normalization
            segment_ids_test=None,
            confidence=confidence
        )
        
        y_pred_all.append(y_pred)
        lower_all.append(lower)
        upper_all.append(upper)
        
        # Update calibration set
        X_cal = np.vstack([X_cal, X_batch])
        y_cal = np.concatenate([y_cal, y_batch])
        
        # Sliding window
        if window_size is not None and len(y_cal) > window_size:
            X_cal = X_cal[-window_size:]
            y_cal = y_cal[-window_size:]
    
    return (np.concatenate(y_pred_all),
            np.concatenate(lower_all),
            np.concatenate(upper_all))

print('Function defined.')

Function defined.


In [3]:
# --- Run 4 configurations ---
results = {}

# 1. Static + Absolute NCM (baseline from Phase4)
y_pred_sa, lower_sa, upper_sa = get_fast_prediction_intervals(
    model, X_cal, y_cal, X_test, confidence=TARGET_COVERAGE
)
results['Static + Absolute NCM'] = (y_pred_sa, lower_sa, upper_sa)
print('1/6 Static + Absolute NCM done')

# 2. Static + Normalized NCM
y_pred_sn, lower_sn, upper_sn = get_normalized_prediction_intervals(
    model, X_cal, y_cal, X_test, confidence=TARGET_COVERAGE
)
results['Static + Normalized NCM'] = (y_pred_sn, lower_sn, upper_sn)
print('2/6 Static + Normalized NCM done')

# 3. Online Expanding + Absolute NCM (from Phase5 - recreate with fast method)
def online_cp_absolute(model, X_stream, y_stream, X_cal_init, y_cal_init,
                        dates_stream, confidence=0.90, window_size=None):
    X_cal = np.copy(X_cal_init)
    y_cal = np.copy(y_cal_init)
    y_pred_all, lower_all, upper_all = [], [], []
    for date in tqdm(sorted(set(dates_stream)), desc='Online Absolute NCM'):
        mask = dates_stream == date
        X_batch = X_stream[mask]
        y_batch = y_stream[mask]
        y_pred, lower, upper = get_fast_prediction_intervals(
            model, X_cal, y_cal, X_batch, confidence=confidence
        )
        y_pred_all.append(y_pred)
        lower_all.append(lower)
        upper_all.append(upper)
        X_cal = np.vstack([X_cal, X_batch])
        y_cal = np.concatenate([y_cal, y_batch])
        if window_size is not None and len(y_cal) > window_size:
            X_cal = X_cal[-window_size:]
            y_cal = y_cal[-window_size:]
    return np.concatenate(y_pred_all), np.concatenate(lower_all), np.concatenate(upper_all)

# 3a. Online Expanding + Absolute
y_pred_oea, lower_oea, upper_oea = online_cp_absolute(
    model, X_test, y_test, X_cal, y_cal, dates_test, TARGET_COVERAGE, window_size=None
)
results['Online Expanding + Absolute NCM'] = (y_pred_oea, lower_oea, upper_oea)
print('3/6 Online Expanding + Absolute done')

# 3b. Online Sliding-14d + Absolute
ws_14d = int(14 * 391)  # ~5474
y_pred_osa, lower_osa, upper_osa = online_cp_absolute(
    model, X_test, y_test, X_cal, y_cal, dates_test, TARGET_COVERAGE, window_size=ws_14d
)
results['Online Sliding-14d + Absolute NCM'] = (y_pred_osa, lower_osa, upper_osa)
print('4/6 Online Sliding-14d + Absolute done')

# 4a. Online Expanding + Normalized NCM
y_pred_oen, lower_oen, upper_oen = online_cp_with_normalized_ncm(
    model, X_test, y_test, X_cal, y_cal, dates_test, TARGET_COVERAGE, window_size=None
)
results['Online Expanding + Normalized NCM'] = (y_pred_oen, lower_oen, upper_oen)
print('5/6 Online Expanding + Normalized done')

# 4b. Online Sliding-14d + Normalized NCM
y_pred_osn, lower_osn, upper_osn = online_cp_with_normalized_ncm(
    model, X_test, y_test, X_cal, y_cal, dates_test, TARGET_COVERAGE, window_size=ws_14d
)
results['Online Sliding-14d + Normalized NCM'] = (y_pred_osn, lower_osn, upper_osn)
print('6/6 Online Sliding-14d + Normalized done')

print('\nAll experiments complete.')

1/6 Static + Absolute NCM done
2/6 Static + Normalized NCM done


Online Absolute NCM: 100%|██████████| 25/25 [00:00<00:00, 203.83it/s]


3/6 Online Expanding + Absolute done


Online Absolute NCM: 100%|██████████| 25/25 [00:00<00:00, 270.97it/s]


4/6 Online Sliding-14d + Absolute done


Online Normalized NCM: 100%|██████████| 25/25 [00:32<00:00,  1.29s/it]


5/6 Online Expanding + Normalized done


Online Normalized NCM: 100%|██████████| 25/25 [00:13<00:00,  1.90it/s]

6/6 Online Sliding-14d + Normalized done

All experiments complete.


In [4]:
# --- Compare all methods ---
print(f"{'Method':<42s}  {'PICP':>7s}  {'MPIW':>9s}  {'Cal.Err':>8s}  {'Winkler':>9s}  {'>=90%':>5s}")
print('-' * 90)

summary_rows = []
for name, (yp, lo, up) in results.items():
    picp = compute_picp(y_test, lo, up)
    mpiw = compute_mpiw(lo, up)
    cal_err = compute_calibration_error(y_test, lo, up, TARGET_COVERAGE)
    winkler = compute_winkler_score(y_test, lo, up, ALPHA)
    
    hit = 'YES' if picp >= 0.90 else 'no'
    print(f"{name:<42s}  {picp:>7.4f}  {mpiw:>9.1f}  {cal_err:>8.4f}  {winkler:>9.1f}  {hit:>5s}")
    
    summary_rows.append({
        'Method': name,
        'PICP': picp,
        'MPIW (s)': mpiw,
        'Cal. Error': cal_err,
        'Winkler': winkler,
        'Coverage >= 90%': picp >= 0.90
    })

summary_df = pd.DataFrame(summary_rows)
print('\n')
display(summary_df.style.format({
    'PICP': '{:.4f}', 'MPIW (s)': '{:.1f}', 'Cal. Error': '{:.4f}', 'Winkler': '{:.1f}'
}).highlight_min(subset=['Cal. Error', 'Winkler'], color='lightgreen')
 .highlight_max(subset=['PICP'], color='lightgreen'))

Method                                         PICP       MPIW   Cal.Err    Winkler  >=90%
------------------------------------------------------------------------------------------
Static + Absolute NCM                        0.8871     3998.7    0.0129     5714.7     no
Static + Normalized NCM                      0.9241     3865.8    0.0241     4878.1    YES
Online Expanding + Absolute NCM              0.8914     4084.5    0.0086     5706.4     no
Online Sliding-14d + Absolute NCM            0.8939     4122.4    0.0061     5706.9     no
Online Expanding + Normalized NCM            0.9115     3744.2    0.0115     4857.2    YES
Online Sliding-14d + Normalized NCM          0.9053     3678.5    0.0053     4848.7    YES




,Method,PICP,MPIW (s),Cal. Error,Winkler,Coverage >= 90%
0,Static + Absolute NCM,0.8871,3998.7,0.0129,5714.7,False
1,Static + Normalized NCM,0.9241,3865.8,0.0241,4878.1,True
2,Online Expanding + Absolute NCM,0.8914,4084.5,0.0086,5706.4,False
3,Online Sliding-14d + Absolute NCM,0.8939,4122.4,0.0061,5706.9,False
4,Online Expanding + Normalized NCM,0.9115,3744.2,0.0115,4857.2,True
5,Online Sliding-14d + Normalized NCM,0.9053,3678.5,0.0053,4848.7,True


## Interpretation

This 2x3 factorial comparison isolates the effects of:
- **Calibration strategy** (Static vs Online Expanding vs Online Sliding-14d)
- **NCM type** (Absolute vs Normalized)

Key questions to answer:
1. Does Normalized NCM improve PICP compared to Absolute NCM (within same calibration strategy)?
2. Does Normalized NCM improve MPIW (narrower intervals for same coverage)?
3. Does the combination (Online + Normalized) reach the 90% target?